#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze Layer

In [0]:
df=spark.table("workspace.bronze.crm_prd_info")

#Data Transformation

#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

## Product Key Parsing

In [0]:
df=df.withColumn("cat_id",F.regexp_replace(F.substring(col("prd_key"),1,5),"-","_"))
df=df.withColumn("prd_key",F.substring(col("prd_key"),7,F.length(col("prd_key"))))

## Cost Cleanup

In [0]:
df =df.withColumn("prd_cost",F.coalesce(col("prd_cost"),F.lit(0)))

## Product Line Normalization

In [0]:
df=(
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line"))=='M','Mountain')
        .when(F.upper(F.col("prd_line"))=='R','Road')
        .when(F.upper(F.col("prd_line"))=='T','Touring')
        .when(F.upper(F.col("prd_line"))=='S','Other Sales')
        .otherwise('n/a')
    )

)


## Date casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

#Renaming columns

In [0]:
rename_cols={
"prd_id":"product_id",
"cat_id":"category_id",
"prd_key":"product_number",
"prd_nm":"product_name",
"prd_cost":"product_cost",
"prd_line":"product_line",
"prd_start_dt":"product_start_date",
"prd_end_dt":"product_end_date"
 }
   

for old_name,new_name in rename_cols.items():
    df=df.withColumnRenamed(old_name,new_name)


#Write to Silver Layer

In [0]:
(
df.write
.mode("overwrite")
.format('delta')
.saveAsTable("silver.crm_products")
)